# 04 — Walk-forward validation

Prequential hybrid: tournament-batched Elo from 2000+, supervised B0/B2 from 2005+.
Primary report: mean log-loss Δ vs B1 on 2019+ `pre_tournament` rows.

In [3]:
from datetime import date

from pathlib import Path



import pandas as pd



from tml.validation import (

    PrequentialConfig,

    common_eligible,

    delta_log_loss,

    final_era_primary_delta,

    moving_block_ci_delta,

    run_prequential,

)



In [4]:
# Load oriented matches from notebook 01 (persisted parquet).

modeling_matches = pd.read_parquet(Path("../data/processed/modeling.parquet"))
# Required by the feature store / lineage fields
if "dataset_snapshot_id" not in modeling_matches.columns:
    modeling_matches["dataset_snapshot_id"] = "notebook-modeling"



# Full 2000+ walk-forward is slow in a notebook. Use a short smoke-style window first.

# For the real primary result, widen years / remove the head filter and expect a long run.

year = modeling_matches["tourney_date"].map(lambda d: pd.Timestamp(d).year)

modeling_matches = modeling_matches.loc[year.between(2018, 2019)].copy()



# Keep a few earliest tournaments per level-year so the cell finishes quickly.

modeling_matches["_year"] = year.loc[modeling_matches.index]

keys = (

    modeling_matches[["_year", "tour_level", "tourney_date", "tourney_id"]]

    .drop_duplicates()

    .sort_values(["_year", "tour_level", "tourney_date", "tourney_id"], kind="stable")

    .groupby(["_year", "tour_level"], sort=False)

    .head(8)

)

modeling_matches = (

    modeling_matches.merge(keys, on=["_year", "tour_level", "tourney_date", "tourney_id"], how="inner")

    .drop(columns=["_year"])

    .reset_index(drop=True)

)

print("rows", len(modeling_matches))



feature_store = Path("../data/processed/notebook_wf_features.parquet")

feature_store.unlink(missing_ok=True)



config = PrequentialConfig(

    burn_in_end=date(2017, 12, 31),

    elo_from=2018,

    supervised_from=2019,

    rolling_years=1,

    dev_era=(2018, 2018),

    final_era=(2019, 2100),

    feature_store_path=feature_store,

)

result = run_prequential(modeling_matches, config=config)

preds = result.predictions

print(preds.shape)

preds.head()



rows 1053


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


(1053, 11)


,match_id,tourney_date,year,p_b0,p_b1,p_b2,y,tour_level,surface,prediction_regime,dataset_snapshot_id
0,atp:2018-339:1,2018-01-01,2018,NaN,0.5,NaN,0,atp,Hard,pre_tournament,notebook-modeling
1,atp:2018-339:2,2018-01-01,2018,NaN,0.5,NaN,0,atp,Hard,pre_tournament,notebook-modeling
2,atp:2018-339:3,2018-01-01,2018,NaN,0.5,NaN,1,atp,Hard,pre_tournament,notebook-modeling
3,atp:2018-339:4,2018-01-01,2018,NaN,0.5,NaN,0,atp,Hard,pre_tournament,notebook-modeling
4,atp:2018-339:5,2018-01-01,2018,NaN,0.5,NaN,1,atp,Hard,pre_tournament,notebook-modeling


In [5]:
# Diagnostic metrics on this smoke window — not the locked 2019+ full-corpus report.

eligible = common_eligible(preds, ("p_b0", "p_b1", "p_b2"))

print("common eligible", len(eligible))

if len(eligible):

    y = eligible["y"].to_numpy(dtype=float)

    print("delta B0-B1", delta_log_loss(y, eligible["p_b0"], eligible["p_b1"]))

    print("delta B2-B1", delta_log_loss(y, eligible["p_b2"], eligible["p_b1"]))

    # Primary helper filters year>=2019 + pre_tournament (works on this window's 2019 rows).

    print(final_era_primary_delta(preds, year_min=2019))

    print("block CI B2-B1", moving_block_ci_delta(eligible, n_boot=100))



common eligible 550
delta B0-B1 -0.05124864570695531
delta B2-B1 -0.03579238249412686
{'year_min': 2019, 'regime': 'pre_tournament', 'n': 550, 'delta_b0': -0.05124864570695531, 'delta_b2': -0.03579238249412686, 'log_loss_b1': 0.6846968108663428, 'log_loss_b0': 0.6334481651593875, 'log_loss_b2': 0.648904428372216, 'brier_b1': 0.24577969453708973, 'brier_b2': 0.2288260654659648, 'calibration_intercept_b2': 0.25051200219584024, 'calibration_slope_b2': 0.7234933497907269}
block CI B2-B1 (-0.24626913063459976, -0.029079240433361186)
